***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [6. 成像中的去卷积](6_0_introduction.ipynb)
    * 上一节： [6.2 点源的迭代去卷积（CLEAN）](6_2_clean.ipynb)
    * 下一节： [6.4 残图与图像质量](6_4_residuals_and_iqa.ipynb)

***


导入标准模块:

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['figure.max_open_warning'] = 0

NOTEBOOK_DIR = Path("6_Deconvolution") if Path("6_Deconvolution").exists() else Path(".")
NOTEBOOK_DIR = NOTEBOOK_DIR.resolve()
STYLE_PATH = NOTEBOOK_DIR.parent / "style" / "course.css"
TOGGLE_PATH = NOTEBOOK_DIR.parent / "style" / "code_toggle.html"

try:
    from IPython.display import HTML
except ImportError:
    def HTML(*args, **kwargs):
        return None

if STYLE_PATH.exists():
    try:
        HTML(STYLE_PATH.read_text(encoding='utf-8'))
    except Exception:
        pass

导入本节所需的专用模块:

In [ ]:
# This notebook now uses self-contained synthetic data examples,
# so optional FITS/aplpy dependencies are not required.

In [ ]:
if TOGGLE_PATH.exists():
    HTML(TOGGLE_PATH.read_text(encoding='utf-8'))

***

## 6.3 CLEAN 的实现方式<a id='deconv:sec:flavours'></a>


上一节我们在点源近似的前提下引入了迭代去卷积，也就是 `CLEAN`，并给出了一个图像域实现示例。随着计算能力的提高，`CLEAN` 已发展出多种变体，以适应不同成像场景。这里我们先介绍几种最基础的实现方式及其优缺点，包括图像域、网格化可见度域以及可见度域 `CLEAN`。为了让 notebook 能独立运行，下面的图像示例采用一套自给自足的合成观测数据，但它们展示的主瓣、旁瓣、残图和复原图之间的关系与真实观测是一致的。回顾结果可以看到，去卷积后我们已经能够得到一幅旁瓣大幅减弱的复原图。

In [ ]:
def generalGauss2d(x0, y0, sigmax, sigmay, amp=1.0, theta=0.0):
    rtheta = np.deg2rad(theta)
    a = (np.cos(rtheta) ** 2.0) / (2.0 * sigmax ** 2.0) + (np.sin(rtheta) ** 2.0) / (2.0 * sigmay ** 2.0)
    b = -(np.sin(2.0 * rtheta)) / (4.0 * sigmax ** 2.0) + (np.sin(2.0 * rtheta)) / (4.0 * sigmay ** 2.0)
    c = (np.sin(rtheta) ** 2.0) / (2.0 * sigmax ** 2.0) + (np.cos(rtheta) ** 2.0) / (2.0 * sigmay ** 2.0)
    return lambda x, y: amp * np.exp(-1.0 * (a * (x - x0) ** 2.0 - 2.0 * b * (x - x0) * (y - y0) + c * (y - y0) ** 2.0))



def centered_gaussian(n, fwhm_x, fwhm_y=None, theta=0.0, amp=1.0):
    if fwhm_y is None:
        fwhm_y = fwhm_x
    sigma_x = fwhm_x / (2.0 * np.sqrt(2.0 * np.log(2.0)))
    sigma_y = fwhm_y / (2.0 * np.sqrt(2.0 * np.log(2.0)))
    yy, xx = np.mgrid[0:n, 0:n].astype(float)
    func = generalGauss2d((n - 1) / 2.0, (n - 1) / 2.0, sigma_x, sigma_y, amp=amp, theta=theta)
    arr = func(xx, yy)
    return arr / arr.max()



def fft_convolve_same(image, kernel):
    return np.real(
        np.fft.fftshift(
            np.fft.ifft2(np.fft.fft2(np.fft.ifftshift(image)) * np.fft.fft2(np.fft.ifftshift(kernel)))
        )
    )



def shift_image(image, dy, dx):
    return np.roll(np.roll(image, int(dy), axis=0), int(dx), axis=1)



def make_uv_dirty_beam(n, n_pairs=260, radius_frac=0.32, seed=0):
    rng = np.random.default_rng(seed)
    center = n // 2
    radius = radius_frac * n
    angles = rng.uniform(0.0, 2.0 * np.pi, n_pairs)
    radii = radius * np.sqrt(rng.uniform(0.02, 1.0, n_pairs))
    u = np.rint(radii * np.cos(angles)).astype(int)
    v = np.rint(radii * np.sin(angles)).astype(int)
    keep = (np.abs(u) < n // 2 - 2) & (np.abs(v) < n // 2 - 2) & ((u != 0) | (v != 0))
    half = np.column_stack([u[keep], v[keep]])
    coords = np.vstack([half, -half])
    sampling = np.zeros((n, n), dtype=float)
    sampling[coords[:, 1] + center, coords[:, 0] + center] = 1.0
    psf = np.real(np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(sampling))))
    psf /= psf.max()
    return sampling, psf, coords



def hogbom_clean(dirty, psf, gain=0.1, niter=300, threshold=0.0, clean_beam=None, capture_iters=(0, 1, 3, 10, 30, 100, 300)):
    capture_iters = set(capture_iters)
    residual = dirty.copy()
    model = np.zeros_like(dirty)
    center = np.array(psf.shape) // 2
    states = {}

    def capture(iteration):
        model_image = fft_convolve_same(model, clean_beam) if clean_beam is not None else model.copy()
        restored = model_image + residual
        states[iteration] = {
            'residual': residual.copy(),
            'model': model.copy(),
            'model_image': model_image.copy(),
            'restored': restored.copy(),
        }

    capture(0)
    last_iter = 0
    for iteration in range(1, niter + 1):
        iy, ix = np.unravel_index(np.argmax(np.abs(residual)), residual.shape)
        peak = residual[iy, ix]
        if np.abs(peak) <= threshold:
            break
        model[iy, ix] += gain * peak
        shifted_psf = shift_image(psf, iy - center[0], ix - center[1])
        residual -= gain * peak * shifted_psf
        last_iter = iteration
        if iteration in capture_iters:
            capture(iteration)
    if last_iter not in states:
        capture(last_iter)
    return {'states': states, 'final_iteration': last_iter, 'model': model, 'residual': residual}


DECONV_N = 96
rng = np.random.default_rng(6)
trueSky = np.zeros((DECONV_N, DECONV_N))
for x0, y0, flux in [(48, 47, 1.00), (63, 56, 0.55), (30, 36, 0.33), (71, 26, 0.20), (25, 70, 0.16)]:
    trueSky[y0, x0] = flux

samplingMask, dirtyPSF, uvCoords = make_uv_dirty_beam(DECONV_N, seed=9)
dirtyImage = fft_convolve_same(trueSky, dirtyPSF) + 0.008 * rng.standard_normal((DECONV_N, DECONV_N))
idealBeam = centered_gaussian(DECONV_N, 2.5)
deconvRun = hogbom_clean(dirtyImage, dirtyPSF, gain=0.12, niter=300, threshold=0.0, clean_beam=idealBeam)
deconvStates = deconvRun['states']
overviewRestored = deconvStates[max(deconvStates.keys())]['restored']
uvExtent = [-DECONV_N / 2, DECONV_N / 2, -DECONV_N / 2, DECONV_N / 2]

fig = plt.figure(figsize=(16, 7))
ax = plt.subplot(121)
ax.set_title('Dirty image')
plt.imshow(dirtyImage, origin='lower', cmap='viridis')
plt.colorbar(shrink=0.75)

ax = plt.subplot(122)
ax.set_title('Restored image after 300 iterations')
plt.imshow(overviewRestored, origin='lower', cmap='viridis')
plt.colorbar(shrink=0.75)
plt.tight_layout()

*图：合成观测得到的脏图（左）与经过 300 次 CLEAN 迭代后的复原图（右）。可以看到复原图中的旁瓣结构明显减弱。*

### 6.3.1 图像域（Högbom）


1974 年，[<cite data-cite='1974A&AS...15..417H'>Högbom</cite> &#10548;](http://adsabs.harvard.edu/abs/1974A%26AS...15..417H) 首次提出了 `CLEAN` 算法；其 notebook 实现可见 [这里 &#10142;](hogbom_clean.ipynb)。该方法的核心思想非常直接：在点源假设和已知 PSF 的前提下，从脏图中不断减去按一定流量缩放后的 PSF，从而逐步移除这些点源，最后得到一个由 $\delta$ 函数组成的天空模型和一幅主要由噪声及未去除源构成的残图。下面给出这种图像域 `CLEAN` 的伪代码。


$\textbf{input: } I^{D}(l,m), \ \textrm{PSF}(l,m), \ \gamma, \ f_{\textrm{thresh}}, \ N$

$\textbf{initialize: } S^{\textrm{model}} \leftarrow \{\}, I^{\textrm{res}} \leftarrow I^{D}, i \leftarrow 0$

$\textbf{while} \ \textrm{any}(I^{\textrm{res}} > f_{\textrm{thresh}}) \ \textrm{or} \ i \leq N \ \textbf{do:}$

$\qquad l_{\textrm{max}}, m_{\textrm{max}} \leftarrow \underset{l,m}{\operatorname{argmax}} I^{\textrm{res}}(l,m)$

$\qquad f_{\textrm{max}} \leftarrow I^{D}(l_{\textrm{max}}, m_{\textrm{max}})$

$\qquad I^{\textrm{res}} \leftarrow I^{\textrm{res}} - \gamma \cdot f_{\textrm{max}} \cdot \textrm{PSF}(l+l_{\textrm{max}}, m+m_{\textrm{max}})$

$\qquad S^{\textrm{model}} \leftarrow S^{\textrm{model}} + \{l_{\textrm{max}}, m_{\textrm{max}}: \gamma \cdot f_{\textrm{max}}\}$

$\qquad i \leftarrow i +1$

$\textbf{ouput: } S^{\textrm{model}}, I^{\textrm{res}}$

输入为脏图和PSF（两者都是$l$和$m$的二维函数）、循环增益参数$\gamma$、流量阈值$f_{\textrm{thresh}}$ 或 最高迭代次数 $N$。

The inputs are the dirty image and the PSF, both of which are 2-d functions in $l$ and $m$, a loop gain parameter $\gamma$, a flux threshold $f_{\textrm{thresh}}$ or maximum number of iterations $N$ as stopping criteria.

#### 输入参数：循环增益$(\gamma)$    Input Parameter: Loop gain $(\gamma)$

参数$\gamma$用来控制每次迭代过程中要减掉的流量比例，取值范围为0到1。如果$\gamma=1$，则该像素上的所有流量将一次性全部减掉，残图里会出现空洞（事实上，每个像素都或多或少含有噪声）。如果$\gamma=0$，则减掉的流量为零，相当于没有去卷积。因此，$\gamma$的取值范围为0到1，一般设置为0.1，即每次迭代，从残图中减去10%的峰值流量。循环增益值设置太大，噪声被减掉导致图像产生“空洞”，会产生过卷积。循环增益设置太小，导致循环次数太多，计算时间过长。

The $\gamma$ parameter ranges from 0 to 1, and controls the amount of flux subtracted with each iteration. If $\gamma=1$ then all the flux of a pixel is subtracted, this is not good as each pixel contains some amount of noise. If $\gamma=0$ then no flux is subtracted and no deconvolution occurs. Thus, $\gamma$ is set somewhere between 0 and 1, a typical value would be $0.1$, this means that each iteration 10% of the peak flux value is subtracted from the residual image. If the loop gain is too high then the deconvolution maybe over ambitious and subtract out noise from the image, leading to 'holes' in the image. If the loop gain is not large enough then the number of iterations required to complete a deconvolution will be prohibitively long.

#### 输入参数：循环停止条件$(N, f_{\textrm{thresh}})$   Input Parameter: Stopping criteria $(N, f_{\textrm{thresh}})$

可以设置两个循环终止条件，当任一条件满足时停止循环。一个条件是迭代次数N，如果设置$N=100$，则迭代100次后，结束去卷积。这个参数的好处在于可以对去卷积的时间消耗进行预估。另一个条件是流量阈值$f_{\textrm{thresh}}$，当残图中剩下的最大流量值小于等于该阈值时，结束去卷积。使用阈值条件的优点在于可以预先知道残图里剩下的最大流量值。

There are two stopping criteria, only one of which is needed. The first is to set the maximum number of iterations, say $N=100$, then after 100 iteration deconvolution will stop. This has the advantage of setting a fixed computation time. The other criteria is to set a flux threshold $f_{\textrm{thresh}}$ such that when the maximum flux value remaining in the residual image is at or below this threshold deconvolution is stopped. This has the advantage of deconvolving down to a known flux level.

#### 初始化和输出 Initialization and Output

初始化过程包括把迭代次数置为0、制作脏图副本以及生成一个空的天空模型。在每次迭代过程中，残图中将被减去一些流量，天空模型里会增加一个$\delta$-function源，源的流量为残图里被减掉的流量。一旦达到结束条件，则输出天空模型和残图。

Deconvolution is initialized by setting the number of iterations to 0, making a copy of the dirty image called the residual image, and creating an empty sky model. During each iteration flux will be subtracted from the residual image and a $\delta$-function source is added to the sky model representing the flux subtracted from the residual image. Once the stopping criteria is met, the completed sky model and residual image is output.

#### 迭代循环  Iterative Loop

只要结束条件不满足，就会执行一次“CLEAN“迭代。首先找出残图中的最大流量$f_{\textrm{max}}$所在的位置$(l_{\textrm{max}}, m_{\textrm{max}})$，把$PSF\cdot\gamma \cdot f_{\textrm{max}}$的中心对齐到残图中的峰值流量位置，PSF乘上脏图上该位置的流量值和循环增益，两者相减使得残图的整体流量变小。

While the stopping criteria has not been met, a `CLEAN` iteration is done. The location $(l_{\textrm{max}}, m_{\textrm{max}})$ of the pixel with the maximum flux $f_{\textrm{max}}$ in the image is found. The PSF is offset to be centred on the peak flux pixel and multiplied by the flux value and gain loop, this is then subtracted from the residual image. The overall flux of the residual image has been reduced.

$$I^{\textrm{res}} \leftarrow I^{\textrm{res}} - \gamma \cdot f_{\textrm{max}} \cdot \textrm{PSF}(l+l_{\textrm{max}}, m+m_{\textrm{max}})$$

接下来，在天空模型里增加一个$\delta$-function的源，源的位置为$(l_{\textrm{max}}, m_{\textrm{max}})$，流量为$\gamma \cdot f_{\textrm{max}}$。

Then a $\delta$-function with flux $\gamma \cdot f_{\textrm{max}}$ source is added to the sky model in the position of the maximum flux.

$$S^{\textrm{model}} \leftarrow S^{\textrm{model}} + \{l_{\textrm{max}}, m_{\textrm{max}}: \gamma \cdot f_{\textrm{max}}\}$$

The new flux at ($l_{\textrm{max}}, m_{\textrm{max}}$) in the sky model is added to any flux in that position from a previous iteration.

这是 `CLEAN` 最原始、也最简单的实现，但正如所料，它也有明显局限。第一，残图与 PSF 都是在固定像素网格上的有限图像；当减去一个偏移后的 PSF 时，实际上只有残图中的一部分像素会被更新。这意味着，要么把 PSF 图像扩到至少两倍大（二维情况下总像素数至少变成四倍），要么只能把去卷积限制在残图中央区域。第二，天空模型的分辨率被像素网格锁死，而真实天体几乎不可能恰好落在某个像素中心，因此会引入额外伪影。正是为了解决这些问题，后续才发展出了更新一代的 `CLEAN` 方法。


### 6.3.2 网格化可见度域（Clark）


1980 年，[<cite data-cite='1980A&A....89..377C'>Clark</cite> &#10548;](http://adsabs.harvard.edu/abs/1980A%26A....89..377C) 对 Högbom 方法做出了关键改进：先在图像域执行一段局部的 Högbom 去卷积，再利用快速傅里叶变换到可见度域做一轮“批处理”式的更新。该方法的实现留作一个[练习 &#10142;](clark_clean_assignment.ipynb)。它包含两层循环：次循环（minor cycle）在图像域找出一组亮像素并构建局部天空模型；主循环（major cycle）则把这个局部天空模型与完整 PSF 卷积后，从残图中整体减去，得到新的残图，并把局部模型并入总天空模型。这样做通常比 Högbom 更快，也允许在更大图像区域上做去卷积。


$\textbf{input: } I^{D}(l,m), \ \textrm{PSF}(l,m), \ \gamma, \ f_{\textrm{thresh}}, \ N$

$\textbf{initialize: } S^{\textrm{model}} \leftarrow \{\}, \ I^{\textrm{res}} \leftarrow I^{D}, \ i \leftarrow 0, \ (\textrm{PSF}_{\textrm{sub}}(l,m), \ R_{\textrm{PSF}}) \leftarrow g(\textrm{PSF}(l,m))$

$\textbf{while} \ \textrm{any}(I^{\textrm{res}} > f_{\textrm{thresh}}) \ \textrm{or} \ i \leq N \ \textbf{do:} \quad [\textrm{Major Cycle}]$

$\qquad l_{\textrm{max}}, m_{\textrm{max}} \leftarrow \underset{l,m}{\operatorname{argmax}} I^{\textrm{res}}(l,m)$

$\qquad f_{\textrm{max}} \leftarrow I^{D}(l_{\textrm{max}}, m_{\textrm{max}})$

$\qquad S^{\textrm{model}}_{\textrm{partial}} \leftarrow \textrm{Hogbom}(I^{\textrm{res}}, \  \textrm{PSF}_{\textrm{sub}}, \ \gamma, \ f_{\textrm{max}} \cdot R_{\textrm{PSF}}) \quad [\textrm{Minor Cycle}]$

$\qquad V^{\textrm{model}}_{\textrm{partial}} \leftarrow \mathscr{F}\{S^{\textrm{model}}_{\textrm{partial}}\}, V^S \leftarrow \mathscr{F}\{\textrm{PSF}\}$

$\qquad I^{\textrm{res}} \leftarrow I^{\textrm{res}} - \mathscr{F}^{-1}\{V^S \cdot V^{\textrm{model}}_{\textrm{partial}}\}$

$\qquad S^{\textrm{model}} \leftarrow S^{\textrm{model}} + S^{\textrm{model}}_{\textrm{partial}}$

$\qquad i \leftarrow i +1$

$\textbf{ouput: } S^{\textrm{model}}, I^{\textrm{res}}$

输入数据和参数与图像域 `CLEAN` 基本相同，只是在主循环中把 Högbom 作为次循环调用。

#### 初始化

初始化阶段多了一步：先从完整 PSF 中选择一个“子 PSF”，供次循环中的图像域 `CLEAN` 使用：

$$\textrm{PSF}_{\textrm{sub}}(l,m), \ R_{\textrm{PSF}} \leftarrow g(\textrm{PSF}(l,m))$$

这个函数 $g$ 并没有唯一标准定义，具体取决于实现。通常会取 PSF 中央主瓣及最高旁瓣附近的一块区域，除非阵列具有很强冗余，此时可能需要更特殊的截断策略。函数同时返回最高旁瓣与主瓣之比 $R_{\textrm{PSF}}$，其值小于 1。这样做的依据是：PSF 的大部分能量都集中在这一块区域中，而且若图像某个像素亮度高于第一旁瓣比例所允许的范围，那么它很可能包含真实源。

子 PSF 越小，可去卷积的图像范围越大，但误差风险也越高；子 PSF 越大，则误差更小，但可处理区域会更受限。因此，子 PSF 大小本质上是一个折中参数。

#### 次循环

次循环的作用，是利用当前残图和子 PSF 在图像域中构造一个局部天空模型：

$$S^{\textrm{model}}_{\textrm{partial}} \leftarrow \textrm{Hogbom}(I^{\textrm{res}}, \  \textrm{PSF}_{\textrm{sub}}, \ \gamma, \ f_{\textrm{max}} \cdot R_{\textrm{PSF}})$$

它的停止阈值取为最大流量 $f_{\textrm{max}}$ 乘以旁瓣比值 $R_{\textrm{PSF}}$。这个比值越大，次循环就能做得越深。由于此时我们只关心得到局部天空模型，所以次循环内部产生的残图通常会被忽略。

#### 主循环

与 Högbom 类似，主循环仍然要先找出残图中的最大流量及其位置。随后调用 Högbom 次循环，在高阈值下生成一个局部天空模型；然后把这个局部模型变换到可见度域，与完整 PSF 做卷积，再变回图像域，从残图中一次性减掉。这样一来，就避免了逐像素逐次小规模减法带来的低效率。


### 6.3.3 可见度域（Cotton-Schwab）

`CLEAN` 最初诞生于计算与内存资源都极其有限的时代，因此早期实现必须非常仔细地权衡这些限制。随着计算能力不断增强，人们得以发展出更先进的去卷积方法。Clark 之后的下一个重要进展来自 1984 年 [<cite data-cite='1984AJ.....89.1076S'>Schwab</cite> &#10548;](http://adsabs.harvard.edu/abs/1984AJ.....89.1076S)，他提出了在**未网格化可见度**上进行去卷积的改进方法。这里的“未网格化可见度”，指的就是观测直接产生、尚未经过 [$\S$ 5.3](../5_Imaging/5_3_gridding_and_degridding.ipynb) 所述网格化处理的原始可见度。Cotton-Schwab 方法能够在整个图像上执行去卷积，同时避免 PSF 伪影混叠，但代价是需要更高的计算量和内存。它至今仍然被广泛使用，也是许多现代 `CLEAN` 实现的基础。像 `CLEAN` 一词本身一样，Cotton-Schwab 在不同软件里也会有不同实现，因此很难写成一个完全抽象且通用的简单伪代码。从大体结构上看，它依然延续了 Clark 的主循环/次循环框架，但主循环里减去的已不再是局部天空模型的网格化可见度，而是根据天空模型直接计算出的未网格化可见度。若源数不多，可以对每个源逐个做直接傅里叶变换；若源数很多，更常见的做法是先把天空模型转换成网格化可见度，再利用[$\S$ 5.3 &#10142;](../5_Imaging/5_3_gridding_and_degridding.ipynb) 介绍的 de-gridder 方法去近似恢复未网格化可见度。


### 6.3.4 理想 PSF 与复原图像


去卷积之后，通常还会生成一张复原图。这就是我们在展示结果时最常看到、也最“好看”的那幅图。需要强调的是：复原图并不会提供比天空模型和残图更多的信息，它更多是一种便于观察和展示的可视化产物。下面给出一个复原图示例。


In [ ]:
fig = plt.figure(figsize=(8, 7))
plt.imshow(overviewRestored, origin='lower', cmap='viridis')
plt.title('Restored image')
plt.colorbar(shrink=0.75)

*图：合成观测经过 CLEAN 去卷积后得到的复原图。*

复原图的生成方式很简单：把天空模型与一个“理想 PSF”或者“理想束”卷积，然后再加上残图。


$$I_{\textrm{restored}} = I_{\textrm{skymodel}} \circ \textrm{PSF}_{\textrm{ideal}} + I_{\textrm{residual}}$$

这里的 $I_{\textrm{skymodel}}$ 是一幅由 $\delta$ 函数组成的天空模型图。所谓理想 PSF，可以有不同定义，但本质上都是某种“理想干涉阵列”的响应。若一个理想阵列能够对最长基线以内的全部空间模式进行采样，那么它会产生一个圆形采样函数，并对应带旁瓣的艾里斑 PSF（见[$\S$ 5.2](../5_Imaging/5_2_sampling_functions_and_psfs.ipynb)）。更理想的情况，则是引入一个平滑 taper，使采样函数连续平滑，从而对应一个没有旁瓣的平滑 PSF，例如高斯。实践中，最常见的做法是对观测 PSF 的主瓣进行二维高斯拟合，并把这个拟合结果作为理想 PSF。高斯的优点在于既容易拟合主瓣，又不会重新引入旁瓣，同时还能保持图像的大体分辨率。当然也可以选用其他形式的理想 PSF，例如直接截取观测 PSF 的主瓣，但多数成像器默认仍使用高斯理想 PSF。其尺度在不同实现中会略有差异，常见做法是拟合一个可旋转二维高斯，或者先计算 PSF 的半高全宽，再构造具有同样半高全宽的二维高斯函数。实际软件通常会把这些理想束参数记录在输出图像的元数据中，用于后续测量与结果解释。


### 6.3.5 一个合成观测的去卷积示例

介绍完不同 `CLEAN` 实现之后，我们再通过一个更直观的例子看看迭代去卷积到底是怎样一步步发生的。这里使用一组带旁瓣的合成脏图，并采用简化的 Högbom `CLEAN` 来演示模型、残图和复原图随迭代如何演化。

In [ ]:
def plotDeconvModelResidual(niter):
    state = deconvStates[niter]
    fig = plt.figure(figsize=(16, 7))

    plt.subplot(121)
    plt.imshow(state['residual'], origin='lower', cmap='viridis')
    plt.title(f'Residual image ({niter} iterations)')
    plt.colorbar(shrink=0.75)

    plt.subplot(122)
    plt.imshow(state['model_image'], origin='lower', cmap='viridis')
    plt.title(f'Clean model convolved with ideal beam ({niter} iterations)')
    plt.colorbar(shrink=0.75)
    plt.tight_layout()

    peak = np.max(np.abs(state['residual']))
    model_flux = np.sum(state['model'])
    print(f'iteration = {niter:3d} | residual peak = {peak:.3f} | accumulated clean flux = {model_flux:.3f}')



def plotDeconvRestored(niter):
    state = deconvStates[niter]
    restored = state['restored']
    restored_vis = np.log10(np.abs(np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(restored)))) + 1e-6)

    fig = plt.figure(figsize=(16, 7))
    plt.subplot(121)
    plt.imshow(restored, origin='lower', cmap='viridis')
    plt.title(f'Restored image ({niter} iterations)')
    plt.colorbar(shrink=0.75)

    plt.subplot(122)
    plt.imshow(restored_vis, origin='lower', extent=uvExtent, cmap='magma')
    plt.scatter(uvCoords[:, 0], uvCoords[:, 1], s=3, c='w')
    plt.title(f'Fourier amplitudes of restored image ({niter} iterations)')
    plt.xlabel('u (grid units)')
    plt.ylabel('v (grid units)')
    plt.tight_layout()

这个例子不依赖外部成像器，而是直接在 notebook 内生成脏图、PSF 和 CLEAN 迭代状态，因此更适合观察算法本身：初始时残图就是脏图，天空模型为空；随后每一步都会从残图里减去一部分主瓣和旁瓣响应，并把对应流量写入模型。

In [ ]:
plotDeconvModelResidual(0)

*图：初始化后的残图与天空模型。初始残图就是脏图，而初始天空模型为空。*

第一次迭代会找到图像中央附近的最亮源。然后把以该位置为中心、按峰值流量和增益缩放后的 PSF 从残图中减去，同时在天空模型对应位置加入一个 $\delta$ 函数。请注意，下图中为了更便于视觉理解，展示的是已经与理想 PSF 卷积后的天空模型。由于一次迭代只移除了一个源的一小部分流量，残图表面上几乎看不出变化，但更新后的天空模型中已经能看到那个源。

In [ ]:
plotDeconvModelResidual(1)

*图：一次 CLEAN 迭代后的残图与天空模型。中央最亮源的一部分流量已被转移到天空模型中，因此残图看起来变化很小。*

当迭代推进到第 10 次时，中央源已有相当一部分流量从残图中移除，旁瓣起伏也随之减弱。不过，中央源依然存在于残图中；与此同时，其他较亮源也开始进入模型。

In [ ]:
plotDeconvModelResidual(10)

*图：10 次 CLEAN 迭代后的残图和天空模型，模型中已经开始出现多个源。*

当迭代达到 100 次时，残图中许多源的流量已明显下降，而天空模型中则出现了多个分量。此时残图中仍然还有可继续去卷积的流量，但随着峰值越来越接近噪声底，继续迭代就会有把噪声也加入天空模型的风险。

In [ ]:
plotDeconvModelResidual(100)

*图：100 次 CLEAN 迭代后的残图和天空模型。多个源已被移入天空模型，但残图中仍剩余部分流量，只是其峰值已接近图像噪声水平。*

迭代继续到 300 次之后，残图基本已经接近噪声状态。虽然仍然存在少量残余流量，但已经不适合继续去卷积。此时得到的天空模型就是最终结果，其中包含多个人工加入的点源分量。

In [ ]:
plotDeconvModelResidual(300)

*图：300 次 CLEAN 迭代后的残图和天空模型。残图已经非常接近噪声，因此必须停止去卷积；最终天空模型显示在右图中。*

我们还可以换个角度看去卷积：不仅看复原图，也看其傅里叶域分布。最开始时，可见度只沿离散的 $uv$ 轨迹被采样，因此观测 PSF 带有明显旁瓣。通过去卷积并生成理想 PSF 下的复原图，实际上相当于在一定程度上“填补”了那些原先没有采样到的可见度区域。只经过少数几次迭代时，复原图看起来仍很像脏图，而傅里叶域中原始的轨迹结构依然非常明显。

In [ ]:
plotDeconvRestored(3)

*图：经过 3 次 CLEAN 迭代后的复原图与傅里叶域分布。复原图仍与脏图相近，而采样轨迹十分清晰。*

经过 30 次迭代后，复原图中的 PSF 旁瓣已经明显减弱。傅里叶图里，原本清晰的 $uv$ 轨迹仍然存在，但其上已经叠加了一层由模型补全出的平滑结构。

In [ ]:
plotDeconvRestored(30)

*图：经过 30 次 CLEAN 迭代后的复原图和傅里叶域分布。复原图中的旁瓣更弱，而傅里叶平面在原始采样轨迹之外也开始出现补全后的结构。*

当迭代达到 300 次时，复原图中已经几乎看不到明显旁瓣，整体效果相当不错。傅里叶图里，最初离散的轨迹仍可辨认，但大多数区域已经被模型补全后的连续结构所覆盖。由于我们选择了高斯作为理想 PSF，因此傅里叶幅度会从中心向外逐渐衰减。

In [ ]:
plotDeconvRestored(300)

*图：经过 300 次 CLEAN 迭代后的复原图和傅里叶域分布。复原图几乎没有明显旁瓣，而傅里叶域也表现出更加充分的采样。*

这里最值得强调的一点是：`CLEAN` 本质上是在用一种迭代预测的方式去填补那些从未真正采样到的可见度。由于我们永远无法知道“完整可见度域”究竟长什么样，因此 `CLEAN` 得到的只是无数可能解中的一种。之所以这类方法常常有效，是因为它建立在“多数源可以近似为点源”的经验性假设之上，而这个假设在很多情况下确实足够好，但也并非始终成立。

### 6.3.6 CLEAN 的局限性以及现代去卷积的发展


再次强调，迭代去卷积并不是完美方法，也不存在一个适用于所有情况的通用“完美去卷积”。不过在大多数实际问题中，它依然是一种相当有效的近似解法。随着计算能力持续提高，以及对更高质量成像的需求不断增长，去卷积方法也在不断发展。

#### 多尺度

标准 Högbom `CLEAN` 假设天空主要由点源组成，因此当目标包含明显展宽结构时，它往往需要用大量相邻点分量去“拼出”扩展源，结果容易留下条纹状残差，也会拖慢收敛速度。针对这个问题， [<cite data-cite='2008ISTSP...2..793C'>Cornwell 2008</cite> &#10548;](http://adsabs.harvard.edu/abs/2008ISTSP...2..793C) 提出了 `multiscale CLEAN`：在每一步迭代时，不仅比较单像素点源模板，还比较一组不同尺度的平滑模板，从而优先用较少的大尺度分量去拟合扩展辐射。

下面的例子用同一组脏图分别执行单尺度 CLEAN 与 multiscale CLEAN。为了把“尺度选择”本身的效果看得更清楚，我们固定两者都做 120 次迭代，并额外估算如果坚持只用点分量，需要多少次迭代才能达到 multiscale 的残图 RMS。

In [ ]:
def make_extended_clean_example(n=96, seed=21):
    yy, xx = np.mgrid[0:n, 0:n].astype(float)
    sky = (
        0.90 * generalGauss2d(48, 44, 6.5, 10.0, amp=1.0, theta=25.0)(xx, yy)
        + 0.28 * generalGauss2d(26, 68, 3.0, 3.0, amp=1.0, theta=0.0)(xx, yy)
    )
    sky[63, 28] += 0.35
    sky[24, 70] += 0.22
    sampling, psf, coords = make_uv_dirty_beam(n, n_pairs=220, radius_frac=0.28, seed=seed)
    rng = np.random.default_rng(seed)
    dirty = fft_convolve_same(sky, psf) + 0.006 * rng.standard_normal((n, n))
    return sky, dirty, psf, coords



def scale_kernel(n, scale):
    if scale == 0:
        kernel = np.zeros((n, n), dtype=float)
        kernel[n // 2, n // 2] = 1.0
        return kernel
    kernel = centered_gaussian(n, scale)
    return kernel / np.sqrt(np.sum(kernel ** 2))



def multiscale_clean(dirty, psf, scales=(0, 4, 10, 18), gain=0.08, niter=120, scale_bias=0.6, clean_beam=None):
    n = dirty.shape[0]
    center = np.array(psf.shape) // 2
    kernels = {scale: scale_kernel(n, scale) for scale in scales}
    kernel_ffts = {scale: np.fft.fft2(np.fft.ifftshift(kernels[scale])) for scale in scales}
    psf_fft = np.fft.fft2(np.fft.ifftshift(psf))
    cross_terms = {
        p: {
            q: np.real(np.fft.fftshift(np.fft.ifft2(kernel_ffts[q] * psf_fft * kernel_ffts[p])))
            for q in scales
        }
        for p in scales
    }
    responses = {scale: fft_convolve_same(dirty, kernels[scale]) for scale in scales}
    response_peaks = {scale: cross_terms[scale][scale][center[0], center[1]] for scale in scales}
    model = np.zeros_like(dirty)
    chosen_scales = []
    max_scale = max(scales) if scales else 0

    for _ in range(niter):
        best = None
        for scale in scales:
            response = responses[scale]
            iy, ix = np.unravel_index(np.argmax(np.abs(response)), response.shape)
            peak = response[iy, ix]
            bias = 1.0 - scale_bias * (scale / max_scale) if max_scale > 0 else 1.0
            score = np.abs(peak) * bias
            if best is None or score > best['score']:
                best = {'scale': scale, 'peak': peak, 'iy': iy, 'ix': ix, 'score': score}

        amp = gain * best['peak'] / max(response_peaks[best['scale']], 1e-8)
        shifted_kernel = shift_image(kernels[best['scale']], best['iy'] - center[0], best['ix'] - center[1])
        model += amp * shifted_kernel
        for scale in scales:
            shifted_term = shift_image(cross_terms[best['scale']][scale], best['iy'] - center[0], best['ix'] - center[1])
            responses[scale] -= amp * shifted_term
        chosen_scales.append(best['scale'])

    residual = dirty - fft_convolve_same(model, psf)
    model_image = fft_convolve_same(model, clean_beam) if clean_beam is not None else model.copy()
    restored = model_image + residual
    return {'model': model, 'residual': residual, 'restored': restored, 'chosen_scales': chosen_scales}


extSky, extDirty, extPSF, extCoords = make_extended_clean_example()
extBeam = centered_gaussian(extSky.shape[0], 3.0)
pointRun = hogbom_clean(extDirty, extPSF, gain=0.12, niter=120, threshold=0.0, clean_beam=extBeam, capture_iters=(120,))
pointResult = pointRun['states'][120]
multiscaleResult = multiscale_clean(extDirty, extPSF, scales=(0, 4, 10, 18), gain=0.08, niter=120, scale_bias=0.6, clean_beam=extBeam)

pointImageRms = np.sqrt(np.mean((pointResult['restored'] - extSky) ** 2))
multiscaleImageRms = np.sqrt(np.mean((multiscaleResult['restored'] - extSky) ** 2))
pointResidualRms = np.sqrt(np.mean(pointResult['residual'] ** 2))
multiscaleResidualRms = np.sqrt(np.mean(multiscaleResult['residual'] ** 2))

pointResidual = extDirty.copy()
center = np.array(extPSF.shape) // 2
pointItersToMatch = None
for iteration in range(1, 1501):
    iy, ix = np.unravel_index(np.argmax(np.abs(pointResidual)), pointResidual.shape)
    peak = pointResidual[iy, ix]
    pointResidual -= 0.12 * peak * shift_image(extPSF, iy - center[0], ix - center[1])
    if np.sqrt(np.mean(pointResidual ** 2)) <= multiscaleResidualRms:
        pointItersToMatch = iteration
        break

scaleCounts = {scale: multiscaleResult['chosen_scales'].count(scale) for scale in sorted(set(multiscaleResult['chosen_scales']))}
print(f"Image RMS after 120 point-CLEAN iterations       = {pointImageRms:.4f}")
print(f"Image RMS after 120 multiscale iterations        = {multiscaleImageRms:.4f}")
print(f"Residual RMS after 120 point-CLEAN iterations    = {pointResidualRms:.4f}")
print(f"Residual RMS after 120 multiscale iterations     = {multiscaleResidualRms:.4f}")
print(f"Point CLEAN iterations needed to match that RMS  = {pointItersToMatch}")
print(f"Chosen multiscale components by scale            = {scaleCounts}")

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes[0, 0].imshow(extSky, origin='lower', cmap='viridis')
axes[0, 0].set_title('True sky')
axes[0, 1].imshow(extDirty, origin='lower', cmap='viridis')
axes[0, 1].set_title('Dirty image')
axes[0, 2].imshow(pointResult['restored'], origin='lower', cmap='viridis')
axes[0, 2].set_title('Point CLEAN (120 iterations)')
axes[1, 0].imshow(pointResult['residual'], origin='lower', cmap='coolwarm')
axes[1, 0].set_title('Point CLEAN residual')
axes[1, 1].imshow(multiscaleResult['restored'], origin='lower', cmap='viridis')
axes[1, 1].set_title('Multiscale CLEAN (120 iterations)')
axes[1, 2].imshow(multiscaleResult['residual'], origin='lower', cmap='coolwarm')
axes[1, 2].set_title('Multiscale residual')
plt.tight_layout()

这个构造例子里会看到一个很直观的结果：在同样 120 次迭代预算下，multiscale CLEAN 的复原图误差和残图 RMS 都明显低于只使用点分量的 Högbom CLEAN；如果坚持只用点分量，本例中通常要迭代到约一千次，残图 RMS 才会接近 multiscale 120 次时的水平。换句话说，多尺度方法的优势不只是“图更平滑”，而是它能用更合适的基函数更快地吸收扩展结构的通量。

#### 多频率

前面已经提到过多频率问题。在 [$\S$ 5.2 &#10142;](../5_Imaging/5_2_sampling_functions_and_psfs.ipynb) 中，我们讨论了通过增大观测带宽来改善 $uv$ 覆盖的方法。但那种说法只在源具有平坦谱指数，或者频带足够窄、可以近似看成平坦谱时才成立。对宽带观测而言，更合理的做法是在去卷积时把每个源同时建模为“流量 + 谱指数”两个参数，而不只拟合流量，这类方法被称为 [<cite data-cite='2011A&A...532A..71R'>multi-frequency synthesis</cite> &#10548;](http://arxiv.org/abs/1106.2745)。

下面的示例并不试图完整重现 MT-MFS 的全部实现细节，而是只演示其最核心的想法。为了把谱建模本身单独拿出来看，这里先假设各频率图像都已经恢复到同一个 restoring beam；在这个前提下，我们比较“直接平均各频率图像”和“用 Taylor 项拟合参考频率亮度与谱指数”两种做法。

In [ ]:
MFS_N = 96
yy, xx = np.mgrid[0:MFS_N, 0:MFS_N].astype(float)
nu0 = 1.0
freqs = np.array([0.85, 0.925, 1.00, 1.075, 1.15])

I0 = np.zeros((MFS_N, MFS_N), dtype=float)
alpha_num = np.zeros_like(I0)

sourceA = generalGauss2d(28, 66, 4.0, 4.0, amp=0.9, theta=0.0)(xx, yy)
sourceB = generalGauss2d(62, 30, 7.0, 10.0, amp=0.6, theta=30.0)(xx, yy)
sourceC = generalGauss2d(70, 70, 2.5, 2.5, amp=0.25, theta=0.0)(xx, yy)

I0 += sourceA + sourceB + sourceC
alpha_num += -0.9 * sourceA + 0.4 * sourceB - 0.2 * sourceC
alphaTrue = np.zeros_like(I0)
mask = I0 > 1e-6
alphaTrue[mask] = alpha_num[mask] / I0[mask]

restoringBeam = centered_gaussian(MFS_N, 4.0)
beamI0 = fft_convolve_same(I0, restoringBeam)
beamAlphaI0 = fft_convolve_same(alphaTrue * I0, restoringBeam)
beamAlphaTrue = np.full_like(I0, np.nan)
validBeam = beamI0 > 0.05 * np.max(beamI0)
beamAlphaTrue[validBeam] = beamAlphaI0[validBeam] / beamI0[validBeam]

rng = np.random.default_rng(3)
observedCube = []
for nu in freqs:
    skyNu = I0 * (nu / nu0) ** alphaTrue
    observedNu = fft_convolve_same(skyNu, restoringBeam) + 0.0010 * rng.standard_normal((MFS_N, MFS_N))
    observedCube.append(observedNu)
observedCube = np.array(observedCube)

flatAverage = np.mean(observedCube, axis=0)
x = np.log(freqs / nu0)
A = np.column_stack([np.ones_like(x), x])
coeff = np.linalg.pinv(A) @ observedCube.reshape(len(freqs), -1)
I0Taylor = coeff[0].reshape(MFS_N, MFS_N)
I1Taylor = coeff[1].reshape(MFS_N, MFS_N)
alphaEst = np.full_like(I0Taylor, np.nan)
valid = I0Taylor > 0.05 * np.max(I0Taylor)
alphaEst[valid] = I1Taylor[valid] / I0Taylor[valid]

maskA = sourceA > 0.20 * np.max(sourceA)
maskB = sourceB > 0.20 * np.max(sourceB)
flatI0Rms = np.sqrt(np.mean((flatAverage - beamI0) ** 2))
taylorI0Rms = np.sqrt(np.mean((I0Taylor - beamI0) ** 2))
alphaRms = np.sqrt(np.nanmean((alphaEst[valid] - beamAlphaTrue[valid]) ** 2))

print(f"Source A peak at nu0 (true I0, beam-smoothed) = {np.max(beamI0[maskA]):.3f}")
print(f"Source A peak from flat-spectrum average      = {np.max(flatAverage[maskA]):.3f}")
print(f"Source A peak from Taylor fit                 = {np.max(I0Taylor[maskA]):.3f}")
print(f"Source B mean spectral index (true)           = {np.nanmean(beamAlphaTrue[maskB & valid]):.3f}")
print(f"Source B mean spectral index (estimated)      = {np.nanmean(alphaEst[maskB & valid]):.3f}")
print(f"I0 RMS from flat-spectrum average             = {flatI0Rms:.4f}")
print(f"I0 RMS from Taylor fit                        = {taylorI0Rms:.4f}")
print(f"Spectral-index RMS on bright pixels           = {alphaRms:.4f}")

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes[0, 0].imshow(beamI0, origin='lower', cmap='viridis')
axes[0, 0].set_title('Beam-smoothed true $I_0$ at $\nu_0$')
axes[0, 1].imshow(flatAverage, origin='lower', cmap='viridis')
axes[0, 1].set_title('Flat-spectrum average image')
axes[0, 2].imshow(I0Taylor, origin='lower', cmap='viridis')
axes[0, 2].set_title('Taylor-term $I_0$ estimate')
axes[1, 0].imshow(beamAlphaTrue, origin='lower', cmap='coolwarm', vmin=-1.0, vmax=1.0)
axes[1, 0].set_title('Beam-smoothed true spectral index')
axes[1, 1].imshow(alphaEst, origin='lower', cmap='coolwarm', vmin=-1.0, vmax=1.0)
axes[1, 1].set_title('Estimated spectral index')
axes[1, 2].imshow(I0Taylor - flatAverage, origin='lower', cmap='coolwarm')
axes[1, 2].set_title('Taylor minus flat-spectrum image')
plt.tight_layout()

在这个简化示例里，各频率图像已经被恢复到同一个 restoring beam，因此图中的差别几乎完全来自谱模型本身。可以看到，简单平均得到的“连续谱图”会把谱指数信息折叠进亮度估计里，因此参考频率下的亮度会带偏；而 Taylor 项拟合则能更接近参考频率亮度图，并在亮源区域恢复出合理的谱指数。这正是现代宽带去卷积器优于“逐频道成图再平均”方法的核心原因之一。真实 MT-MFS 还需要把这个谱模型与频率相关的 $uv$ 覆盖、PSF 和去卷积过程耦合起来，但基本思想与这里的演示一致。

#### 宽场近似

根据 van Cittert-Zernike 定理，可见度域和图像域之间的傅里叶关系只在小角近似下才严格成立。通过付出更多计算量，我们可以像 [$\S$ 5.5 &#10142;](../5_Imaging/5_5_widefield_effect.ipynb) 所述那样，对 $w$ 项做改正来改善这一近似。很多现代成像器和去卷积算法都已经默认提供了这类宽场校正选项。`CLEAN` 的一个局限在于，它在次循环中通常使用的是指向相位中心的单一 PSF，而真实 PSF 会随方向发生变化。因此，对远离相位中心的源进行去卷积时，本应使用不同 PSF，否则一个点源就可能被错误拆成多个分量。对此，常见改进方向包括 facet 成像、W-projection / AW-projection，以及更一般的方向依赖去卷积；可参考 [<cite data-cite='1992A&A...261..353C'>Radio-interferometric imaging of very large fields - The problem of non-coplanar arrays</cite> &#10548;](http://adsabs.harvard.edu/abs/1992A&A...261..353C) 与 [<cite data-cite='2014MNRAS.444..606O'>WSCLEAN</cite> &#10548;](http://arxiv.org/abs/1407.1943)。

***

* 下一节： [6.4 残图与图像质量](6_4_residuals_and_iqa.ipynb)
